In [212]:
import pandas as pd
import numpy as np
import datetime

from pathlib import Path 

In [213]:
current_dir = Path(Path.cwd()).parent
data_dir = current_dir / "data"
source_data_dir = data_dir / "source"
print(data_dir)

c:\Users\pedro\DEV\dengue_prediction\dengue_prediction\data


In [214]:

import re
import unicodedata
 
def show_columns(df, n_range = 7):
    df = list(df.columns)
    l = []
    range = n_range
    for c in df:
        if range == n_range:
            print(l)
            l = []
            range = 0
        range += 1
        l.append(c)
    print("\n")

def normalize_column_name(col):
    # tira acentos
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    # minúsculas
    col = col.lower()
    # troca qualquer coisa que não seja letra ou número por _
    col = re.sub(r"[^a-z0-9]+", "_", col)
    # remove _ no começo e no fim
    col = col.strip("_")
    return col

# DEGBR

In [215]:

def prep_DEGBR(df:pd.DataFrame, df_cod_municip:pd.DataFrame):
    # id municipio para municipio e UF
    # id_municip -> códigos IBGE
    df = df.merge(
        df_cod_municip,
        left_on="ID_MUNICIP",
        right_on="codigo_municipio",
        how="left"
    )
    # filtra colunas e renomea-as
    df = df[["DT_NOTIFIC", "uf", "nome"]].rename(columns={
        "DT_NOTIFIC": "data",
        "nome": "municipio"
    })
    # garantir o formato
    df["data"] = pd.to_datetime(df["data"], errors="coerce").dt.normalize()
    # Remove colunas com valores ausentes
    df = df.dropna()
    # agrupar
    df = df.groupby(list(df.columns)).size().reset_index(name="casos_dengue")
    # Normalizacao nome colunas 
    df.columns = [normalize_column_name(col) for col in df.columns]
    # Normalizaco colunas nao numericas
    df["uf"] = df["uf"].astype(str).str.strip().str.upper()
    df["municipio"] = df["municipio"].astype(str).str.strip().str.lower()   
    return df 

def get_municip_map_df(cod_munipc_uniques):
    # load csv
    municipios = pd.read_csv(
        source_data_dir / "municipios.csv",
        usecols=["codigo_ibge", "nome", "codigo_uf"],
        sep=","
    )

    # load csv
    estados = pd.read_csv(
        source_data_dir / "estados.csv",
        usecols=["codigo_uf", "uf"],
        sep=","
    )
    
    municipios = municipios.merge(
        estados,
        on="codigo_uf",
        how="left"
    )
    # Remove coluna
    municipios = municipios.drop(columns=["codigo_uf"])
    
    # Ajusta e filtra
    municipios["codigo_municipio"] = municipios["codigo_ibge"] // 10
    municipios = municipios.drop(columns=["codigo_ibge"])
    municipios_filtrados = municipios[
        municipios["codigo_municipio"].isin(cod_munipc_uniques)
    ].copy()
    
    return municipios_filtrados

In [216]:
dengBr_dir = source_data_dir / "DENGBR" 
dengBr23_file = dengBr_dir / "DENGBR23.csv"
dengBr24_file = dengBr_dir / "DENGBR24.csv"
dengBr25_file = dengBr_dir / "DENGBR25.csv"

dengBr23 = pd.read_csv(dengBr23_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")
dengBr24 = pd.read_csv(dengBr24_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")
dengBr25 = pd.read_csv(dengBr25_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")

cod_munipc_uniques = dengBr23["ID_MUNICIP"].unique().tolist()
cod_munipc_uniques.extend(dengBr24["ID_MUNICIP"].unique().tolist())
cod_munipc_uniques.extend(dengBr25["ID_MUNICIP"].unique().tolist())

df_cod_municip = get_municip_map_df(cod_munipc_uniques)

dengBr23 = prep_DEGBR(dengBr23, df_cod_municip)
dengBr24 = prep_DEGBR(dengBr24, df_cod_municip)
dengBr25 = prep_DEGBR(dengBr25, df_cod_municip)

df_DEGBR = pd.concat([dengBr23, dengBr24, dengBr25], ignore_index=True)

# INMET

In [217]:
def prep_INMET(df: pd.DataFrame, file_name):
    # Remove coluna vazia gerada por ; no final da linha
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")].copy()

    # Remove coluna com muitos valores ausentes
    df = df.drop(
        columns=[
            col for col in df.columns
            if "RADIACAO GLOBAL" in col
        ],
        errors="ignore"
    )

    # Converte a coluna de data
    df["Data"] = pd.to_datetime(
        df["Data"],
        format="%Y/%m/%d",
        errors="coerce"
    )

    # Colunas numéricas
    cols_nao_numericas = ["Data", "Hora UTC"]

    num_cols = [
        col for col in df.columns
        if col not in cols_nao_numericas
    ]

    # Converte colunas numéricas
    for col in num_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )

        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Preenche ausentes com média móvel
    df[num_cols] = df[num_cols].fillna(
        df[num_cols].rolling(window=4, min_periods=2).mean()
    )
    # Remove linhas com valroes ausentes
    df = df.dropna()
    # Agrega por dia
    df = (
        df
        .groupby("Data", as_index=False)[num_cols]
        .mean()
    )

    # Pega UF e município
    partes_nome = file_name.split("_")

    uf = partes_nome[2]
    municipio = partes_nome[4]

    df["uf"] = uf.upper()
    df["municipio"] = municipio.lower()

    # Renomeia colunas
    df.columns = [normalize_column_name(col) for col in df.columns]

    return df
    

In [218]:
dfs_diarios = []
inmet_dir = source_data_dir / "INMET"

for year in [2023, 2024, 2025]:
    year_dir = inmet_dir / str(year)

    for path in year_dir.iterdir():
        file_name = path.stem
        df = pd.read_csv(
            path,
            encoding="latin1",
            sep=";",
            skiprows=8
        )
        df = prep_INMET(df, file_name=file_name)
        dfs_diarios.append(df)
        
df_INMET = pd.concat(dfs_diarios, ignore_index=True)

## media dos 30 dias anteriores

In [ ]:
id_cols = ["uf", "municipio", "data"]

days_mean = df_INMET.groupby(id_cols, as_index=False).mean(numeric_only=True)

days_mean = days_mean.sort_values(["uf", "municipio", "data"])

num_cols = [
    col for col in days_mean.columns
    if col not in id_cols
]

df_INMET_mean_30 = days_mean[id_cols].copy()

for col in num_cols:
    # shift(1) to dont get the current day, and rolling(30) to get the mean of the last 30 days
    df_INMET_mean_30[col] = (
        days_mean
        .groupby(["uf", "municipio"])[col]
        .transform(lambda x: x.shift(1).rolling(window=30, min_periods=30).mean())
    )

## serie temporal dos 30 dias anteriores

# CREAT DATASET (MERGE)

In [220]:
df = df_INMET_mean_30.merge(
    dengBr23,
    on=["data", "municipio", "uf"],
    how="left"
)

# Linhas em que não encontrou informação de dengue
sem_dengue = df[df["casos_dengue"].isna()].copy()

# Municípios sem informação de dengue em pelo menos uma data
municipios_sem_dengue = (
    sem_dengue[["uf", "municipio"]]
    .drop_duplicates()
    .sort_values(["uf", "municipio"])
)

print(municipios_sem_dengue)
print("Qtd municípios sem informação:", municipios_sem_dengue.shape[0])

        uf                  municipio
0       AC             epitaciolandia
6       AC                      feijo
564     AC  parque estadual chandless
731     AC                 rio branco
1827    AL                  arapiraca
...     ..                        ...
391715  TO                      peixe
392282  TO                       pium
392691  TO                   rio sono
392781  TO       santa fe do araguaia
393877  TO    santa rosa do tocantins

[566 rows x 2 columns]
Qtd municípios sem informação: 566


In [221]:
exemplos_com_dengue_por_municipio = (
    df
    .dropna(subset=["casos_dengue"])
    .groupby(["uf", "municipio"])
    .size()
    .reset_index(name="qtd_exemplos_com_dengue")
    .sort_values("qtd_exemplos_com_dengue", ascending=False)
)

print(exemplos_com_dengue_por_municipio)

max_exemplos = exemplos_com_dengue_por_municipio["qtd_exemplos_com_dengue"].max()

bins = [
    0,
    max_exemplos * 0.25,
    max_exemplos * 0.50,
    max_exemplos * 0.75,
    max_exemplos + 1
]

labels = [
    "menos_de_1_quarto",
    "entre_1_e_2_quartos",
    "entre_2_e_3_quartos",
    "entre_3_e_4_quartos"
]

exemplos_com_dengue_por_municipio["faixa_qtd_exemplos"] = pd.cut(
    exemplos_com_dengue_por_municipio["qtd_exemplos_com_dengue"],
    bins=bins,
    labels=labels,
    right=False
)

resumo_faixas = (
    exemplos_com_dengue_por_municipio
    .groupby("faixa_qtd_exemplos", observed=False)
    .size()
    .reset_index(name="qtd_municipios")
)

print("Máximo de exemplos em um município:", max_exemplos)
print(resumo_faixas)

     uf                  municipio  qtd_exemplos_com_dengue
50   ES                 vila velha                      377
46   ES                   linhares                      370
32   BA                   salvador                      363
37   CE                  fortaleza                      348
157  PI                   teresina                      345
..   ..                        ...                      ...
168  PR                   ventania                        1
131  PA               porto de moz                        1
132  PA  santa maria das barreiras                        1
212  SC                  urussanga                        1
191  RS                dom pedrito                        1

[249 rows x 3 columns]
Máximo de exemplos em um município: 377
    faixa_qtd_exemplos  qtd_municipios
0    menos_de_1_quarto             168
1  entre_1_e_2_quartos              46
2  entre_2_e_3_quartos              18
3  entre_3_e_4_quartos              17


In [224]:
df = df_INMET_mean_30.merge(
    df_DEGBR,
    on=["data", "municipio", "uf"],
    how="left"
)

print(df.shape)

# Conta exemplos com dengue por município
exemplos_por_municipio = (
    df
    .dropna(subset=["casos_dengue"])
    .groupby(["uf", "municipio"])
    .size()
    .reset_index(name="qtd_exemplos_com_dengue")
)

# Calcula metade do máximo
max_exemplos = exemplos_por_municipio["qtd_exemplos_com_dengue"].max()
limite_minimo = max_exemplos / 2

# Municípios que têm mais da metade do máximo
municipios_validos = exemplos_por_municipio[
    exemplos_por_municipio["qtd_exemplos_com_dengue"] >= limite_minimo
][["uf", "municipio"]]

print("Máximo de exemplos:", max_exemplos)
print("Limite mínimo:", limite_minimo)
print("Qtd municípios mantidos:", municipios_validos.shape[0])

# Mantém só esses municípios
df = df.merge(
    municipios_validos,
    on=["uf", "municipio"],
    how="inner"
)

# Remove as linhas sem target e os primeiros 30 dias para cada municipio (ja que pefamos a medias dos 30 dias anterires e so temos essa informacoa a partir de 30 dias)
df = df.dropna()

# Data para numérico
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["dia"] = df["data"].dt.day
df["dia_da_semana"] = df["data"].dt.dayofweek

df = df.drop(columns=["data"])
df.to_csv(data_dir/"processed/datasets/casos_de_dengue_dataset.csv", index=False)
print(df)

(395011, 20)
Máximo de exemplos: 1105
Limite mínimo: 552.5
Qtd municípios mantidos: 28
       uf            municipio  precipitacao_total_horario_mm  \
30     AC           rio branco                       0.242222   
31     AC           rio branco                       0.250000   
32     AC           rio branco                       0.250278   
33     AC           rio branco                       0.269722   
34     AC           rio branco                       0.268056   
...    ..                  ...                            ...   
26542  SP  presidente prudente                       0.426481   
26543  SP  presidente prudente                       0.426481   
26544  SP  presidente prudente                       0.426481   
26545  SP  presidente prudente                       0.426481   
26546  SP  presidente prudente                       0.429815   

       pressao_atmosferica_ao_nivel_da_estacao_horaria_mb  \
30                                            992.428472    
31        